In [1]:
import os
import random
import numpy as np
import pandas as pd
from dataset_module import dataset_module

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)

def add_wind_noise(df):
    df = df.copy()
    eps_e, eps_n = np.zeros(len(df)), np.zeros(len(df))
    
    # Parameters
    phi = 0.9       # AR(1) coefficient
    rel_err = 0.15  # 15% RMS forecast error

    # Error std proportional to total wind speed
    sigma = rel_err * df['u'].values / np.sqrt(2)

    # Initialize from stationary distribution
    eps_e[0] = np.random.normal(0, sigma[0])
    eps_n[0] = np.random.normal(0, sigma[0])

    # Generate AR(1) forecast errors
    for i in range(1, len(df)):
        innov_std = sigma[i] * np.sqrt(1 - phi**2)
        eps_e[i] = phi * eps_e[i-1] + np.random.normal(0, innov_std)
        eps_n[i] = phi * eps_n[i-1] + np.random.normal(0, innov_std)

    # Add forecast errors
    df['u_e'] = df['u_e'] + eps_e
    df['u_n'] = df['u_n'] + eps_n
    df['u'] = np.sqrt(df['u_e']**2 + df['u_n']**2)

    return df

# ----- Dataset module -----
cols, df, _, _ = dataset_module()
split = pd.to_datetime('2023-07-01 00:00:00')
df_test = df[df['TmStamp'] > split].reset_index(drop=True)

# Add wind noise
df_fcst = add_wind_noise(df_test)
df_fcst.to_csv('M01_fcst.csv', index=False)

In [2]:
# Compute u_e error
err_e = df_fcst['u_e'] - df_test['u_e']
print(f'Mean u_e error: {np.mean(err_e):.3f}')
print(f'Std of u_e error: {np.std(err_e, ddof=1):.3f}')

# Compute u_n error
err_n = df_fcst['u_n'] - df_test['u_n']
print(f'\nMean u_n error: {np.mean(err_n):.3f}')
print(f'Std of u_n error: {np.std(err_n, ddof=1):.3f}')

# Compute wind speed error
err_u = df_fcst['u'] - df_test['u']
print(f'\nMean wind speed error: {np.mean(err_u):.3f}')
print(f'Std of wind speed error: {np.std(err_u, ddof=1):.3f}')

# Relative RMS wind vector error
err_vec = np.sqrt(err_e**2 + err_n**2)
rel = err_vec / df_test['u']
rms_rel = np.sqrt(np.mean(rel**2))
print(f'\nRelative RMS wind vector error: {100*rms_rel:.2f}%')

Mean u_e error: -0.008
Std of u_e error: 0.779

Mean u_n error: 0.003
Std of u_n error: 0.772

Mean wind speed error: 0.072
Std of wind speed error: 0.764

Relative RMS wind vector error: 17.21%
